In [1]:
# In[1]:


# # Composite DNA Decoder: Training & Evaluation - Eta-Based Variable Ratio
# ## Cross-Platform Robustness Study: Nanopore (R21, B22, NP22, NPF22) + Newer Illumina (BOS22)
# ## Each profile uses its standard sequence length from the corresponding original dataset

# =============================================================================
# CELL 1: DEVICE CONFIGURATION
# =============================================================================
import os
import torch

DEVICE_ID = "2"
os.environ["CUDA_VISIBLE_DEVICES"] = DEVICE_ID


# In[2]:

# =============================================================================
# CELL 2: IMPORTS
# =============================================================================
import random
import pickle
import json
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
import time
from datetime import datetime

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Using device: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")

✅ Using device: cuda
   GPU: NVIDIA GeForce RTX 3080


In [2]:
# In[3]:

# =============================================================================
# CELL 3: CONFIGURATION & HYPERPARAMETERS
# =============================================================================

# ------------------- SELECT ERROR MODEL -------------------
# NEW PLATFORM OPTIONS (cross-platform robustness study):
#   "R21"    -> Oxford Nanopore MinION + Twist (Rang et al. 2021)
#   "B22"    -> Nanopore MinION Short + Twist  (Bar-Lev et al. 2022)
#   "BOS22"  -> Illumina MiSeq 2022 + Twist   (very low error, newer Illumina)
#   "NP22"   -> Nanopore Pilot Nov-2022 + Twist (highly non-uniform across bases)
#   "NPF22"  -> Nanopore Full Pool Nov-2022 + Twist (comprehensive Nanopore)
# ----------------------------------------------------------
ERROR_MODEL = "R21"  # <-- CHANGE THIS

# ------------------- ETA-BASED ALPHABET PARAMETERS -------------------
ETA = 0.2
ELL_VALUES = [-2, -1, 0, 1, 2]

# Dataset parameters
NUM_SAMPLES = 100000
MAX_COVERAGE = 50

# Calculate vocab size
NUM_PURE_BASES = 4
NUM_TWO_MIX_PAIRS = 6
VOCAB_SIZE = NUM_PURE_BASES + NUM_TWO_MIX_PAIRS * len(ELL_VALUES)  # 34

# Error model specifications - standard sequence lengths from original datasets
# All new profiles use the same Erlich/Twist 152nt oligo pool (16nt index → n=136)
# Combined with original profiles (EZ17 n=136, G15 n=104, O17 n=77), this gives
# standard sequence lengths n ∈ {77, 104, 136} across the full set of 8 profiles.
ERROR_MODEL_SPECS = {
    "R21": {
        "full_length": 152,
        "index_length": 16,
        "seq_length": 136,  # 152 - 16 (same Erlich/Twist pool)
        "name": "R21",
        "platform": "Oxford Nanopore MinION",
        "synthesis": "Twist Bioscience"
    },
    "B22": {
        "full_length": 152,
        "index_length": 16,
        "seq_length": 136,  # 152 - 16 (same Erlich/Twist pool)
        "name": "B22",
        "platform": "Nanopore MinION Short",
        "synthesis": "Twist Bioscience"
    },
    "BOS22": {
        "full_length": 152,
        "index_length": 16,
        "seq_length": 136,  # 152 - 16 (same Erlich/Twist pool)
        "name": "BOS22",
        "platform": "Illumina MiSeq 2022",
        "synthesis": "Twist Bioscience"
    },
    "NP22": {
        "full_length": 152,
        "index_length": 16,
        "seq_length": 136,  # 152 - 16 (same Erlich/Twist pool)
        "name": "NP22",
        "platform": "Nanopore Pilot Nov-2022",
        "synthesis": "Twist Bioscience"
    },
    "NPF22": {
        "full_length": 152,
        "index_length": 16,
        "seq_length": 136,  # 152 - 16 (same Erlich/Twist pool)
        "name": "NPF22",
        "platform": "Nanopore Full Pool Nov-2022",
        "synthesis": "Twist Bioscience"
    },
}

# Build configuration
CONFIG = {
    # Error Model
    "error_model": ERROR_MODEL,
    "error_name": ERROR_MODEL_SPECS[ERROR_MODEL]["name"],
    "platform": ERROR_MODEL_SPECS[ERROR_MODEL]["platform"],
    
    # Eta Parameters
    "eta": ETA,
    "ell_values": ELL_VALUES,
    "alphabet_mode": f"eta{ETA}",
    
    # Data Paths (matches dataset_generator_eta_cross_platform.py output)
    "dataset_dir": "./dataset_cross_platform",
    "dataset_name": f"dna_{ERROR_MODEL_SPECS[ERROR_MODEL]['name']}_eta{ETA}",
    
    # Results directory
    "results_dir": f"./results_crossplatform_{ERROR_MODEL_SPECS[ERROR_MODEL]['name']}_eta{ETA}",
    
    # Vocabulary
    "vocab_size": VOCAB_SIZE,
    
    # Sequence Parameters
    "seq_length": ERROR_MODEL_SPECS[ERROR_MODEL]["seq_length"],
    
    # Experiment Parameters
    "coverage_levels": [1, 2, 3, 5, 8, 10, 15, 20, 25, 30, 40, 50],
    
    # Model Architecture (same as original for fair comparison)
    "input_channels": 4,
    "hidden_dim": 128,
    "num_layers": 2,
    "dropout": 0.2,
    "bidirectional": True,
    
    # Training Parameters
    "batch_size": 500,
    "learning_rate": 1e-3,
    "weight_decay": 1e-4,
    "epochs": 100,
    "patience": 10,
    "warmup_epochs": 10,
    "min_lr": 1e-6,
    
    # Reproducibility
    "seed": 42
}

# Complete dataset path
CONFIG["dataset_path"] = (f"{CONFIG['dataset_dir']}/"
                          f"{CONFIG['dataset_name']}_"
                          f"{NUM_SAMPLES}_{MAX_COVERAGE}.pkl")

# Create results directory
os.makedirs(CONFIG['results_dir'], exist_ok=True)

print(f"{'='*70}")
print(f"📋 CROSS-PLATFORM ETA-BASED CONFIGURATION")
print(f"{'='*70}")
print(f"   Error Model: {CONFIG['error_model']} ({CONFIG['error_name']})")
print(f"   Platform: {CONFIG['platform']}")
print(f"   Oligo: {ERROR_MODEL_SPECS[ERROR_MODEL]['full_length']}nt − "
      f"{ERROR_MODEL_SPECS[ERROR_MODEL]['index_length']}nt = "
      f"{CONFIG['seq_length']}nt (standard)")
print(f"   Eta: {CONFIG['eta']}, Ell Values: {CONFIG['ell_values']}")
print(f"   Sequence Length: {CONFIG['seq_length']}")
print(f"   Vocab Size: {CONFIG['vocab_size']} classes")
print(f"   Theoretical Capacity: {np.log2(CONFIG['vocab_size']):.4f} bits/position")
print(f"   Dataset Path: {CONFIG['dataset_path']}")
print(f"   Results Dir: {CONFIG['results_dir']}")
print(f"{'='*70}")


📋 CROSS-PLATFORM ETA-BASED CONFIGURATION
   Error Model: R21 (R21)
   Platform: Oxford Nanopore MinION
   Oligo: 152nt − 16nt = 136nt (standard)
   Eta: 0.2, Ell Values: [-2, -1, 0, 1, 2]
   Sequence Length: 136
   Vocab Size: 34 classes
   Theoretical Capacity: 5.0875 bits/position
   Dataset Path: ./dataset_cross_platform/dna_R21_eta0.2_100000_50.pkl
   Results Dir: ./results_crossplatform_R21_eta0.2


In [3]:
# In[4]:

# =============================================================================
# CELL 4: SEED & REPRODUCIBILITY
# =============================================================================
def set_seed(seed):
    """Set seed for reproducibility across all libraries."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(CONFIG['seed'])
print(f"🎲 Random seed set to: {CONFIG['seed']}")


🎲 Random seed set to: 42


In [4]:
# In[5]:

# =============================================================================
# CELL 5: BUILD ETA-BASED ALPHABET MAPPINGS
# =============================================================================
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
# SAME AS: train_evaluate_eta_based-Erlich.py → Cell 5
# Copy: build_eta_based_symbol_to_idx(), build_eta_based_ideal_vectors()
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

def build_eta_based_symbol_to_idx(eta, ell_values):
    """Build symbol-to-index mapping for eta-based alphabet."""
    symbol_to_idx = {'A': 0, 'C': 1, 'G': 2, 'T': 3}
    
    pair_names = ['B1', 'B2', 'B3', 'B4', 'B5', 'B6']
    current_idx = 4
    
    for pair_name in pair_names:
        for ell in ell_values:
            if ell >= 0:
                symbol_name = f"{pair_name}_ell{ell}"
            else:
                symbol_name = f"{pair_name}_ell_neg{abs(ell)}"
            symbol_to_idx[symbol_name] = current_idx
            current_idx += 1
    
    return symbol_to_idx


def build_eta_based_ideal_vectors(eta, ell_values):
    """
    Build ideal frequency vectors for eta-based alphabet.
    Returns: torch.Tensor of shape (vocab_size, 4)
    """
    ideal_vectors = [
        [1.0, 0.0, 0.0, 0.0],  # A
        [0.0, 1.0, 0.0, 0.0],  # C
        [0.0, 0.0, 1.0, 0.0],  # G
        [0.0, 0.0, 0.0, 1.0],  # T
    ]
    
    # Two-mix pairs: (vec_idx1, vec_idx2)
    pair_indices = [
        (0, 1),  # B1: A|C
        (0, 2),  # B2: A|G
        (0, 3),  # B3: A|T
        (1, 2),  # B4: C|G
        (1, 3),  # B5: C|T
        (2, 3),  # B6: G|T
    ]
    
    for idx1, idx2 in pair_indices:
        for ell in ell_values:
            prob1 = 0.5 + ell * eta
            prob2 = 0.5 - ell * eta
            
            vec = [0.0, 0.0, 0.0, 0.0]
            vec[idx1] = prob1
            vec[idx2] = prob2
            ideal_vectors.append(vec)
    
    return torch.tensor(ideal_vectors, dtype=torch.float32)


# Build mappings
SYMBOL_TO_IDX = build_eta_based_symbol_to_idx(CONFIG["eta"], CONFIG["ell_values"])
IDX_TO_SYMBOL = {v: k for k, v in SYMBOL_TO_IDX.items()}
IDEAL_VECTORS = build_eta_based_ideal_vectors(CONFIG["eta"], CONFIG["ell_values"]).to(device)

print(f"\n📊 Symbol Mappings (η={CONFIG['eta']}):")
print(f"   Total symbols: {len(SYMBOL_TO_IDX)}")
print(f"\n   Sample mappings (first 10):")
for i, (sym, idx) in enumerate(sorted(SYMBOL_TO_IDX.items(), key=lambda x: x[1])[:10]):
    vec = IDEAL_VECTORS[idx].cpu().numpy()
    print(f"   {sym:<16} {idx:<4} [{vec[0]:.2f}, {vec[1]:.2f}, {vec[2]:.2f}, {vec[3]:.2f}]")
print(f"   ... ({len(SYMBOL_TO_IDX) - 10} more)")



📊 Symbol Mappings (η=0.2):
   Total symbols: 34

   Sample mappings (first 10):
   A                0    [1.00, 0.00, 0.00, 0.00]
   C                1    [0.00, 1.00, 0.00, 0.00]
   G                2    [0.00, 0.00, 1.00, 0.00]
   T                3    [0.00, 0.00, 0.00, 1.00]
   B1_ell_neg2      4    [0.10, 0.90, 0.00, 0.00]
   B1_ell_neg1      5    [0.30, 0.70, 0.00, 0.00]
   B1_ell0          6    [0.50, 0.50, 0.00, 0.00]
   B1_ell1          7    [0.70, 0.30, 0.00, 0.00]
   B1_ell2          8    [0.90, 0.10, 0.00, 0.00]
   B2_ell_neg2      9    [0.10, 0.00, 0.90, 0.00]
   ... (24 more)


In [5]:
# In[6]:

# =============================================================================
# CELL 6: DATA PREPROCESSING
# =============================================================================
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
# SAME AS: train_evaluate_eta_based-Erlich.py → Cell 6
# Copy: preprocess_cluster_to_matrix()
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

def preprocess_cluster_to_matrix(cluster_reads, target_length):
    """
    Convert variable-length noisy reads into a (4, target_length) normalized frequency matrix.
    """
    profile_matrix = np.zeros((4, target_length), dtype=np.float32)
    base_map = {'A': 0, 'C': 1, 'G': 2, 'T': 3}
    num_reads = len(cluster_reads)
    
    for read in cluster_reads:
        read_len = len(read)
        if read_len == 0:
            continue
            
        for t_idx in range(target_length):
            read_idx = int((t_idx + 0.5) * (read_len / target_length))
            if read_idx >= read_len:
                read_idx = read_len - 1
            
            base = read[read_idx]
            if base in base_map:
                row_idx = base_map[base]
                profile_matrix[row_idx, t_idx] += 1.0
                
    if num_reads > 0:
        profile_matrix /= num_reads
        
    return profile_matrix


In [6]:
# In[7]:

# =============================================================================
# CELL 7: PYTORCH DATASET CLASS
# =============================================================================
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
# SAME AS: train_evaluate_eta_based-Erlich.py → Cell 7
# Copy: CompositeDNADatasetEta class
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

class CompositeDNADatasetEta(Dataset):
    """PyTorch Dataset for Eta-Based Composite DNA data."""
    
    def __init__(self, data_path, seq_length, symbol_to_idx, limit_coverage=None):
        with open(data_path, 'rb') as f:
            raw_data = pickle.load(f)
        self.samples = raw_data['data']
        self.metadata = raw_data['metadata']
        self.seq_length = seq_length
        self.symbol_to_idx = symbol_to_idx
        self.limit_coverage = limit_coverage
        
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        item = self.samples[idx]
        cluster = item['cluster']
        
        if self.limit_coverage is not None:
            actual_limit = min(self.limit_coverage, len(cluster))
            cluster = cluster[:actual_limit]
            
        x_data = preprocess_cluster_to_matrix(cluster, self.seq_length)
        label_seq = item['label']
        y_data = np.array([self.symbol_to_idx[s] for s in label_seq], dtype=np.longlong)
        
        return torch.tensor(x_data, dtype=torch.float32), torch.tensor(y_data, dtype=torch.long)


In [7]:
# In[8]:

# =============================================================================
# CELL 8: NEURAL NETWORK MODEL
# =============================================================================
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
# SAME AS: train_evaluate_eta_based-Erlich.py → Cell 8
# Copy: CompositeDecoderLSTM class
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

class CompositeDecoderLSTM(nn.Module):
    """Bidirectional LSTM Decoder for Composite DNA."""
    
    def __init__(self, config):
        super(CompositeDecoderLSTM, self).__init__()
        
        self.lstm = nn.LSTM(
            input_size=config['input_channels'],
            hidden_size=config['hidden_dim'],
            num_layers=config['num_layers'],
            batch_first=True,
            bidirectional=config['bidirectional'],
            dropout=config['dropout'] if config['num_layers'] > 1 else 0
        )
        
        fc_in = config['hidden_dim'] * 2 if config['bidirectional'] else config['hidden_dim']
        self.fc = nn.Linear(fc_in, config['vocab_size'])
        
    def forward(self, x):
        # x: (Batch, 4, L) -> (Batch, L, 4)
        x = x.permute(0, 2, 1)
        out, _ = self.lstm(x)
        logits = self.fc(out)
        # Return: (Batch, vocab_size, L)
        return logits.permute(0, 2, 1)

In [8]:
# In[9]:

# =============================================================================
# CELL 9: BASELINE DECODERS
# =============================================================================
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
# SAME AS: train_evaluate_eta_based-Erlich.py → Cell 9
# Copy: min_distance_decoder(), kl_divergence_decoder(), maximum_likelihood_decoder()
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

def min_distance_decoder(obs, ideal_vectors):
    """Minimum Euclidean Distance Decoder (L2 norm)."""
    dists = torch.sum((obs.unsqueeze(2) - ideal_vectors.unsqueeze(0).unsqueeze(0)) ** 2, dim=3)
    return torch.argmin(dists, dim=2)


def kl_divergence_decoder(obs, ideal_vectors, epsilon=0.01):
    """KL Divergence Decoder."""
    ideal_safe = ideal_vectors.clone()
    ideal_safe = torch.clamp(ideal_safe, min=epsilon)
    ideal_safe = ideal_safe / ideal_safe.sum(dim=-1, keepdim=True)
    
    obs_expanded = obs.unsqueeze(2)
    log_ideal = torch.log(ideal_safe).unsqueeze(0).unsqueeze(0)
    
    cross_entropy = -(obs_expanded * log_ideal).sum(dim=-1)
    return torch.argmin(cross_entropy, dim=-1)


def maximum_likelihood_decoder(obs, ideal_vectors, epsilon=0.01):
    """Maximum Likelihood Decoder."""
    ideal_safe = ideal_vectors.clone()
    ideal_safe = torch.clamp(ideal_safe, min=epsilon)
    ideal_safe = ideal_safe / ideal_safe.sum(dim=-1, keepdim=True)
    
    obs_expanded = obs.unsqueeze(2)
    log_ideal = torch.log(ideal_safe).unsqueeze(0).unsqueeze(0)
    
    log_likelihood = (obs_expanded * log_ideal).sum(dim=-1)
    return torch.argmax(log_likelihood, dim=-1)



In [9]:
# In[10]:

# =============================================================================
# CELL 10: EARLY STOPPING CLASS
# =============================================================================
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
# SAME AS: train_evaluate_eta_based-Erlich.py → Cell 10
# Copy: EarlyStopping class
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

class EarlyStopping:
    """Early stopping with patience and best model saving."""
    
    def __init__(self, patience=5, path='checkpoint.pt', verbose=True):
        self.patience = patience
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.path = path
        self.verbose = verbose
        self.best_val_loss = float('inf')

    def __call__(self, val_loss, model):
        score = -val_loss
        
        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
        elif score < self.best_score:
            self.counter += 1
            if self.verbose:
                print(f"      EarlyStopping counter: {self.counter}/{self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
            self.counter = 0
            
    def save_checkpoint(self, val_loss, model):
        if self.verbose:
            print(f"      ✓ Val loss improved ({self.best_val_loss:.4f} → {val_loss:.4f}). Saving...")
        torch.save(model.state_dict(), self.path)
        self.best_val_loss = val_loss


In [10]:
# In[11]:

# =============================================================================
# CELL 11: TRAINING FUNCTION
# =============================================================================
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
# SAME AS: train_evaluate_eta_based-Erlich.py → Cell 11
# Copy: train_model() function
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

def train_model(model, train_loader, val_loader, config, weights_path, device):
    """Train the model with warmup + cosine annealing scheduler."""
    
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(
        model.parameters(), 
        lr=config['learning_rate'],
        weight_decay=config['weight_decay']
    )
    
    warmup_scheduler = LinearLR(optimizer, start_factor=0.1, total_iters=config['warmup_epochs'])
    cosine_scheduler = CosineAnnealingLR(
        optimizer, T_max=config['epochs'] - config['warmup_epochs'], eta_min=config['min_lr']
    )
    scheduler = SequentialLR(
        optimizer, schedulers=[warmup_scheduler, cosine_scheduler], milestones=[config['warmup_epochs']]
    )
    
    early_stopper = EarlyStopping(patience=config['patience'], path=weights_path, verbose=True)
    
    history = {'train_loss': [], 'val_loss': [], 'lr': []}
    
    print(f"\n   🏋️ Training Configuration:")
    print(f"      Epochs: {config['epochs']}, Patience: {config['patience']}")
    print(f"      Warmup: {config['warmup_epochs']} epochs")
    print(f"      LR: {config['learning_rate']} → {config['min_lr']}")
    
    for epoch in range(config['epochs']):
        start_time = time.time()
        
        # Training
        model.train()
        train_loss_accum = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_loss_accum += loss.item()
        avg_train_loss = train_loss_accum / len(train_loader)
        
        # Validation
        model.eval()
        val_loss_accum = 0.0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                val_loss_accum += criterion(outputs, labels).item()
        avg_val_loss = val_loss_accum / len(val_loader)
        
        current_lr = optimizer.param_groups[0]['lr']
        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(avg_val_loss)
        history['lr'].append(current_lr)
        
        elapsed = time.time() - start_time
        print(f"   Epoch {epoch+1:03d}/{config['epochs']} | "
              f"Train: {avg_train_loss:.4f} | Val: {avg_val_loss:.4f} | "
              f"LR: {current_lr:.2e} | Time: {elapsed:.1f}s")
        
        scheduler.step()
        early_stopper(avg_val_loss, model)
        
        if early_stopper.early_stop:
            print(f"\n   🛑 Early stopping triggered at epoch {epoch+1}")
            break
    
    return history


In [11]:
# In[12]:

# =============================================================================
# CELL 12: EVALUATION FUNCTION
# =============================================================================
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
# SAME AS: train_evaluate_eta_based-Erlich.py → Cell 12
# Copy: evaluate_all_decoders() function
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

def evaluate_all_decoders(model, loader, ideal_vectors, device):
    """Evaluate all 4 decoders on the given data loader."""
    model.eval()
    
    correct = {'lstm': 0, 'mindist': 0, 'kl': 0, 'ml': 0}
    total = 0
    
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            obs = inputs.permute(0, 2, 1)
            
            outputs = model(inputs)
            pred_lstm = torch.argmax(outputs, dim=1)
            pred_mindist = min_distance_decoder(obs, ideal_vectors)
            pred_kl = kl_divergence_decoder(obs, ideal_vectors)
            pred_ml = maximum_likelihood_decoder(obs, ideal_vectors)
            
            total += labels.numel()
            correct['lstm'] += (pred_lstm == labels).sum().item()
            correct['mindist'] += (pred_mindist == labels).sum().item()
            correct['kl'] += (pred_kl == labels).sum().item()
            correct['ml'] += (pred_ml == labels).sum().item()
    
    accuracies = {k: 100 * v / total for k, v in correct.items()}
    return accuracies


In [12]:
# In[13]:

# =============================================================================
# CELL 13: FULL EXPERIMENT FOR SINGLE COVERAGE
# =============================================================================

def run_experiment_for_coverage(coverage_M, config, symbol_to_idx, ideal_vectors, device):
    """Run complete experiment for a single coverage level."""
    
    print(f"\n{'='*70}")
    print(f"🔬 EXPERIMENT FOR COVERAGE M = {coverage_M}")
    print(f"   Error Model: {config['error_name']} ({config['platform']})")
    print(f"   Eta: {config['eta']}, Vocab Size: {config['vocab_size']}")
    print(f"   Seq Length: {config['seq_length']}")
    print(f"{'='*70}")
    
    set_seed(config['seed'])
    
    full_ds = CompositeDNADatasetEta(
        config['dataset_path'], config['seq_length'], symbol_to_idx, limit_coverage=coverage_M
    )
    
    train_size = int(0.8 * len(full_ds))
    val_size = len(full_ds) - train_size
    train_ds, val_ds = random_split(full_ds, [train_size, val_size])
    
    train_loader = DataLoader(train_ds, batch_size=config['batch_size'], shuffle=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=config['batch_size'], shuffle=False, num_workers=0)
    
    print(f"   📊 Data: {train_size:,} train | {val_size:,} validation")
    
    model = CompositeDecoderLSTM(config).to(device)
    num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"   🧠 Model: {config['vocab_size']} classes, {num_params:,} parameters")
    
    model_prefix = f"{config['error_name']}_eta{config['eta']}"
    best_weights_path = os.path.join(config['results_dir'], f"best_model_{model_prefix}_M{coverage_M}.pth")
    final_weights_path = os.path.join(config['results_dir'], f"final_model_{model_prefix}_M{coverage_M}.pth")
    history_path = os.path.join(config['results_dir'], f"training_history_{model_prefix}_M{coverage_M}.json")
    
    history = train_model(model, train_loader, val_loader, config, best_weights_path, device)
    
    torch.save(model.state_dict(), final_weights_path)
    print(f"   💾 Final model saved: {final_weights_path}")
    
    with open(history_path, 'w') as f:
        json.dump(history, f, indent=4)
    
    print(f"\n   📈 Evaluating all decoders...")
    model.load_state_dict(torch.load(best_weights_path, map_location=device))
    
    accuracies = evaluate_all_decoders(model, val_loader, ideal_vectors, device)
    
    print(f"\n   ✅ RESULTS M={coverage_M} ({config['error_name']}, η={config['eta']}):")
    print(f"      Bi-LSTM:         {accuracies['lstm']:.2f}%")
    print(f"      Min. Distance:   {accuracies['mindist']:.2f}%")
    print(f"      KL Divergence:   {accuracies['kl']:.2f}%")
    print(f"      Max. Likelihood: {accuracies['ml']:.2f}%")
    
    return accuracies, history

In [13]:
# In[14]:

# =============================================================================
# CELL 14: PLOTTING FUNCTIONS
# =============================================================================
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
# SAME AS: train_evaluate_eta_based-Erlich.py → Cell 14
# Copy: plot_training_history(), plot_comparison_results()
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

def plot_training_history(history, coverage_M, save_path, config):
    """Plot training and validation loss curves."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    epochs = range(1, len(history['train_loss']) + 1)
    
    ax1.plot(epochs, history['train_loss'], 'b-', linewidth=2, label='Train Loss')
    ax1.plot(epochs, history['val_loss'], 'r-', linewidth=2, label='Val Loss')
    ax1.set_xlabel('Epoch', fontsize=12)
    ax1.set_ylabel('Loss', fontsize=12)
    ax1.set_title(f'Training & Validation Loss (M={coverage_M}, η={config["eta"]})\n'
                  f'{config["error_name"]} ({config["platform"]})', fontsize=13)
    ax1.legend(fontsize=11)
    ax1.grid(True, alpha=0.3)
    
    ax2.plot(epochs, history['lr'], 'g-', linewidth=2)
    ax2.set_xlabel('Epoch', fontsize=12)
    ax2.set_ylabel('Learning Rate', fontsize=12)
    ax2.set_title(f'Learning Rate Schedule (M={coverage_M})', fontsize=14)
    ax2.set_yscale('log')
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"   📈 Training plot saved: {save_path}")


def plot_comparison_results(results, save_path, config):
    """Plot comparison of all decoders across coverage levels."""
    plt.figure(figsize=(12, 7))
    
    plt.plot(results['coverage'], results['lstm'], 
             'o-', lw=2.5, ms=8, c='#2ecc71', label='Bi-LSTM (Ours)')
    plt.plot(results['coverage'], results['mindist'], 
             's--', lw=2.5, ms=8, c='#e74c3c', label='Min. Distance')
    plt.plot(results['coverage'], results['kl'], 
             '^-.', lw=2.5, ms=8, c='#3498db', label='KL Divergence')
    plt.plot(results['coverage'], results['ml'], 
             'd:', lw=2.5, ms=8, c='#9b59b6', label='Max. Likelihood')
    
    title = (f"Composite DNA Decoding: η={config['eta']} ({config['vocab_size']} classes)\n"
             f"Error Model: {config['error_name']} ({config['platform']}), "
             f"Seq Length: {config['seq_length']}")
    
    plt.xlabel("Coverage Depth (M)", fontsize=12)
    plt.ylabel("Symbol Accuracy (%)", fontsize=12)
    plt.title(title, fontsize=14)
    plt.legend(fontsize=11, loc='lower right')
    plt.grid(True, alpha=0.3)
    plt.ylim(0, 105)
    plt.xticks(results['coverage'])
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"📈 Comparison plot saved: {save_path}")


In [14]:
# In[15]:

# =============================================================================
# CELL 15: VERIFY DATASET EXISTS
# =============================================================================

print("\n" + "="*70)
print("📦 LOADING DATASET")
print("="*70)

if not os.path.exists(CONFIG['dataset_path']):
    raise FileNotFoundError(
        f"\n❌ Dataset not found: {CONFIG['dataset_path']}\n"
        f"   Please run dataset_generator_eta_cross_platform.py first with:\n"
        f"   ERROR_MODEL = \"{CONFIG['error_model']}\"\n"
        f"   ETA = {CONFIG['eta']}, ELL_VALUES = {CONFIG['ell_values']}"
    )

with open(CONFIG['dataset_path'], 'rb') as f:
    data = pickle.load(f)

print(f"✅ Dataset loaded: {CONFIG['dataset_path']}")
print(f"   Samples: {len(data['data']):,}")
print(f"   Vocab Size: {data['metadata']['vocab_size']}")
print(f"   Eta: {data['metadata']['eta']}")
print(f"   Sequence Length: {data['metadata']['seq_length']}")
if 'platform' in data['metadata']:
    print(f"   Platform: {data['metadata']['platform']}")



📦 LOADING DATASET
✅ Dataset loaded: ./dataset_cross_platform/dna_R21_eta0.2_100000_50.pkl
   Samples: 100,000
   Vocab Size: 34
   Eta: 0.2
   Sequence Length: 136
   Platform: Oxford Nanopore MinION


In [15]:
# In[16]:

# =============================================================================
# CELL 16: MAIN EXECUTION - RUN ALL EXPERIMENTS
# =============================================================================

print("\n" + "="*70)
print("🚀 RUNNING EXPERIMENTS FOR ALL COVERAGE LEVELS")
print("="*70)
print(f"   Error Model: {CONFIG['error_name']} ({CONFIG['platform']})")
print(f"   Eta: {CONFIG['eta']}, Vocab Size: {CONFIG['vocab_size']}")
print(f"   Seq Length: {CONFIG['seq_length']}")
print(f"   Coverage Levels: {CONFIG['coverage_levels']}")

results = {
    'coverage': CONFIG['coverage_levels'],
    'lstm': [],
    'mindist': [],
    'kl': [],
    'ml': [],
    'config': {
        'error_model': CONFIG['error_model'],
        'error_name': CONFIG['error_name'],
        'platform': CONFIG['platform'],
        'seq_length': CONFIG['seq_length'],
        'eta': CONFIG['eta'],
        'ell_values': CONFIG['ell_values'],
        'vocab_size': CONFIG['vocab_size'],
        'hidden_dim': CONFIG['hidden_dim'],
        'num_layers': CONFIG['num_layers'],
        'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }
}

all_histories = {}

for M in CONFIG['coverage_levels']:
    accuracies, history = run_experiment_for_coverage(
        M, CONFIG, SYMBOL_TO_IDX, IDEAL_VECTORS, device
    )
    
    results['lstm'].append(accuracies['lstm'])
    results['mindist'].append(accuracies['mindist'])
    results['kl'].append(accuracies['kl'])
    results['ml'].append(accuracies['ml'])
    all_histories[M] = history
    
    plot_prefix = f"{CONFIG['error_name']}_eta{CONFIG['eta']}"
    plot_path = os.path.join(CONFIG['results_dir'], f"training_plot_{plot_prefix}_M{M}.png")
    plot_training_history(history, M, plot_path, CONFIG)



🚀 RUNNING EXPERIMENTS FOR ALL COVERAGE LEVELS
   Error Model: R21 (Oxford Nanopore MinION)
   Eta: 0.2, Vocab Size: 34
   Seq Length: 136
   Coverage Levels: [1, 2, 3, 5, 8, 10, 15, 20, 25, 30, 40, 50]

🔬 EXPERIMENT FOR COVERAGE M = 1
   Error Model: R21 (Oxford Nanopore MinION)
   Eta: 0.2, Vocab Size: 34
   Seq Length: 136
   📊 Data: 80,000 train | 20,000 validation
   🧠 Model: 34 classes, 541,218 parameters

   🏋️ Training Configuration:
      Epochs: 100, Patience: 10
      Warmup: 10 epochs
      LR: 0.001 → 1e-06
   Epoch 001/100 | Train: 3.4859 | Val: 3.3746 | LR: 1.00e-04 | Time: 40.6s
      ✓ Val loss improved (inf → 3.3746). Saving...
   Epoch 002/100 | Train: 3.2704 | Val: 3.2311 | LR: 1.90e-04 | Time: 42.0s
      ✓ Val loss improved (3.3746 → 3.2311). Saving...
   Epoch 003/100 | Train: 3.2271 | Val: 3.2099 | LR: 2.80e-04 | Time: 40.4s
      ✓ Val loss improved (3.2311 → 3.2099). Saving...
   Epoch 004/100 | Train: 3.2046 | Val: 3.1892 | LR: 3.70e-04 | Time: 40.2s
      ✓ 

/homes/shubham/anaconda3/envs/pytorchenv/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:149: UserWarning: The epoch parameter in `scheduler.step()` was not necessary and is being deprecated where possible. Please use `scheduler.step()` to step the scheduler. During the deprecation, if epoch is different from None, the closed form is used instead of the new chainable form, where available. Please open an issue if you are unable to replicate your use case: https://github.com/pytorch/pytorch/issues/new/choose.
  warnings.warn(EPOCH_DEPRECATION_WARNING, UserWarning)


   Epoch 011/100 | Train: 3.1774 | Val: 3.1745 | LR: 1.00e-03 | Time: 39.9s
      EarlyStopping counter: 1/10
   Epoch 012/100 | Train: 3.1768 | Val: 3.1739 | LR: 1.00e-03 | Time: 39.0s
      ✓ Val loss improved (3.1744 → 3.1739). Saving...
   Epoch 013/100 | Train: 3.1764 | Val: 3.1735 | LR: 9.99e-04 | Time: 39.3s
      ✓ Val loss improved (3.1739 → 3.1735). Saving...
   Epoch 014/100 | Train: 3.1760 | Val: 3.1735 | LR: 9.97e-04 | Time: 40.0s
      EarlyStopping counter: 1/10
   Epoch 015/100 | Train: 3.1756 | Val: 3.1736 | LR: 9.95e-04 | Time: 39.4s
      EarlyStopping counter: 2/10
   Epoch 016/100 | Train: 3.1754 | Val: 3.1741 | LR: 9.92e-04 | Time: 39.3s
      EarlyStopping counter: 3/10
   Epoch 017/100 | Train: 3.1753 | Val: 3.1731 | LR: 9.89e-04 | Time: 38.2s
      ✓ Val loss improved (3.1735 → 3.1731). Saving...
   Epoch 018/100 | Train: 3.1750 | Val: 3.1731 | LR: 9.85e-04 | Time: 39.6s
      ✓ Val loss improved (3.1731 → 3.1731). Saving...
   Epoch 019/100 | Train: 3.1749 | V

   Epoch 029/100 | Train: 2.9107 | Val: 2.9086 | LR: 9.05e-04 | Time: 65.8s
      ✓ Val loss improved (2.9089 → 2.9086). Saving...
   Epoch 030/100 | Train: 2.9098 | Val: 2.9074 | LR: 8.94e-04 | Time: 64.9s
      ✓ Val loss improved (2.9086 → 2.9074). Saving...
   Epoch 031/100 | Train: 2.9092 | Val: 2.9071 | LR: 8.83e-04 | Time: 64.6s
      ✓ Val loss improved (2.9074 → 2.9071). Saving...
   Epoch 032/100 | Train: 2.9084 | Val: 2.9072 | LR: 8.72e-04 | Time: 65.6s
      EarlyStopping counter: 1/10
   Epoch 033/100 | Train: 2.9076 | Val: 2.9066 | LR: 8.60e-04 | Time: 65.2s
      ✓ Val loss improved (2.9071 → 2.9066). Saving...
   Epoch 034/100 | Train: 2.9071 | Val: 2.9066 | LR: 8.47e-04 | Time: 64.7s
      EarlyStopping counter: 1/10
   Epoch 035/100 | Train: 2.9065 | Val: 2.9049 | LR: 8.35e-04 | Time: 64.5s
      ✓ Val loss improved (2.9066 → 2.9049). Saving...
   Epoch 036/100 | Train: 2.9056 | Val: 2.9062 | LR: 8.22e-04 | Time: 65.9s
      EarlyStopping counter: 1/10
   Epoch 037/10

   Epoch 003/100 | Train: 2.9616 | Val: 2.9262 | LR: 2.80e-04 | Time: 95.9s
      ✓ Val loss improved (2.9764 → 2.9262). Saving...
   Epoch 004/100 | Train: 2.9076 | Val: 2.8782 | LR: 3.70e-04 | Time: 92.4s
      ✓ Val loss improved (2.9262 → 2.8782). Saving...
   Epoch 005/100 | Train: 2.8759 | Val: 2.8568 | LR: 4.60e-04 | Time: 91.6s
      ✓ Val loss improved (2.8782 → 2.8568). Saving...
   Epoch 006/100 | Train: 2.8588 | Val: 2.8434 | LR: 5.50e-04 | Time: 91.4s
      ✓ Val loss improved (2.8568 → 2.8434). Saving...
   Epoch 007/100 | Train: 2.8475 | Val: 2.8329 | LR: 6.40e-04 | Time: 92.4s
      ✓ Val loss improved (2.8434 → 2.8329). Saving...
   Epoch 008/100 | Train: 2.8358 | Val: 2.8238 | LR: 7.30e-04 | Time: 91.6s
      ✓ Val loss improved (2.8329 → 2.8238). Saving...
   Epoch 009/100 | Train: 2.8280 | Val: 2.8169 | LR: 8.20e-04 | Time: 91.5s
      ✓ Val loss improved (2.8238 → 2.8169). Saving...
   Epoch 010/100 | Train: 2.8223 | Val: 2.8121 | LR: 9.10e-04 | Time: 93.0s
      ✓

...

   📈 Training plot saved: ./results_crossplatform_R21_eta0.2/training_plot_R21_eta0.2_M25.png

🔬 EXPERIMENT FOR COVERAGE M = 30
   Error Model: R21 (Oxford Nanopore MinION)
   Eta: 0.2, Vocab Size: 34
   Seq Length: 136
   📊 Data: 80,000 train | 20,000 validation
   🧠 Model: 34 classes, 541,218 parameters

   🏋️ Training Configuration:
      Epochs: 100, Patience: 10
      Warmup: 10 epochs
      LR: 0.001 → 1e-06
   Epoch 001/100 | Train: 3.4747 | Val: 3.3093 | LR: 1.00e-04 | Time: 1175.6s
      ✓ Val loss improved (inf → 3.3093). Saving...
   Epoch 002/100 | Train: 2.9183 | Val: 2.4888 | LR: 1.90e-04 | Time: 1037.9s
      ✓ Val loss improved (3.3093 → 2.4888). Saving...
   Epoch 003/100 | Train: 2.2884 | Val: 2.1221 | LR: 2.80e-04 | Time: 834.0s
      ✓ Val loss improved (2.4888 → 2.1221). Saving...
   Epoch 004/100 | Train: 2.0811 | Val: 1.9944 | LR: 3.70e-04 | Time: 826.1s
      ✓ Val loss improved (2.1221 → 1.9944). Saving...
   Epoch 005/100 | Train: 1.9820 | Val: 1.9138 | LR: 4.

   Epoch 061/100 | Train: 1.5097 | Val: 1.5023 | LR: 4.14e-04 | Time: 818.6s
      EarlyStopping counter: 1/10
   Epoch 062/100 | Train: 1.5098 | Val: 1.4993 | LR: 3.97e-04 | Time: 819.1s
      ✓ Val loss improved (1.5013 → 1.4993). Saving...
   Epoch 063/100 | Train: 1.5081 | Val: 1.4979 | LR: 3.80e-04 | Time: 818.5s
      ✓ Val loss improved (1.4993 → 1.4979). Saving...
   Epoch 064/100 | Train: 1.5086 | Val: 1.4982 | LR: 3.63e-04 | Time: 888.5s
      EarlyStopping counter: 1/10
   Epoch 065/100 | Train: 1.5066 | Val: 1.4971 | LR: 3.46e-04 | Time: 1118.7s
      ✓ Val loss improved (1.4979 → 1.4971). Saving...
   Epoch 066/100 | Train: 1.5062 | Val: 1.4967 | LR: 3.30e-04 | Time: 1056.8s
      ✓ Val loss improved (1.4971 → 1.4967). Saving...
   Epoch 067/100 | Train: 1.5059 | Val: 1.4971 | LR: 3.13e-04 | Time: 834.4s
      EarlyStopping counter: 1/10
   Epoch 068/100 | Train: 1.5050 | Val: 1.4972 | LR: 2.97e-04 | Time: 821.7s
      EarlyStopping counter: 2/10
   Epoch 069/100 | Train: 

   Epoch 019/100 | Train: 1.4429 | Val: 1.4102 | LR: 9.81e-04 | Time: 1076.2s
      ✓ Val loss improved (1.4189 → 1.4102). Saving...
   Epoch 020/100 | Train: 1.4353 | Val: 1.4123 | LR: 9.76e-04 | Time: 1072.8s
      EarlyStopping counter: 1/10
   Epoch 021/100 | Train: 1.4299 | Val: 1.4041 | LR: 9.70e-04 | Time: 1075.9s
      ✓ Val loss improved (1.4102 → 1.4041). Saving...
   Epoch 022/100 | Train: 1.4240 | Val: 1.3967 | LR: 9.64e-04 | Time: 1083.4s
      ✓ Val loss improved (1.4041 → 1.3967). Saving...
   Epoch 023/100 | Train: 1.4185 | Val: 1.3918 | LR: 9.57e-04 | Time: 1074.6s
      ✓ Val loss improved (1.3967 → 1.3918). Saving...
   Epoch 024/100 | Train: 1.4135 | Val: 1.3900 | LR: 9.49e-04 | Time: 1083.7s
      ✓ Val loss improved (1.3918 → 1.3900). Saving...
   Epoch 025/100 | Train: 1.4098 | Val: 1.3842 | LR: 9.42e-04 | Time: 1072.6s
      ✓ Val loss improved (1.3900 → 1.3842). Saving...
   Epoch 026/100 | Train: 1.4063 | Val: 1.3852 | LR: 9.33e-04 | Time: 1074.8s
      EarlyS

   Epoch 083/100 | Train: 1.3302 | Val: 1.3198 | LR: 9.64e-05 | Time: 1079.5s
      ✓ Val loss improved (1.3204 → 1.3198). Saving...
   Epoch 084/100 | Train: 1.3299 | Val: 1.3212 | LR: 8.64e-05 | Time: 1105.7s
      EarlyStopping counter: 1/10
   Epoch 085/100 | Train: 1.3295 | Val: 1.3195 | LR: 7.69e-05 | Time: 1068.2s
      ✓ Val loss improved (1.3198 → 1.3195). Saving...
   Epoch 086/100 | Train: 1.3292 | Val: 1.3196 | LR: 6.79e-05 | Time: 1063.9s
      EarlyStopping counter: 1/10
   Epoch 087/100 | Train: 1.3291 | Val: 1.3193 | LR: 5.95e-05 | Time: 1062.9s
      ✓ Val loss improved (1.3195 → 1.3193). Saving...
   Epoch 088/100 | Train: 1.3289 | Val: 1.3193 | LR: 5.16e-05 | Time: 1065.4s
      ✓ Val loss improved (1.3193 → 1.3193). Saving...
   Epoch 089/100 | Train: 1.3285 | Val: 1.3191 | LR: 4.42e-05 | Time: 1062.7s
      ✓ Val loss improved (1.3193 → 1.3191). Saving...
   Epoch 090/100 | Train: 1.3285 | Val: 1.3189 | LR: 3.74e-05 | Time: 1063.1s
      ✓ Val loss improved (1.3191

   Epoch 040/100 | Train: 1.2412 | Val: 1.2195 | LR: 7.65e-04 | Time: 1318.4s
      ✓ Val loss improved (1.2207 → 1.2195). Saving...
   Epoch 041/100 | Train: 1.2399 | Val: 1.2185 | LR: 7.50e-04 | Time: 1319.0s
      ✓ Val loss improved (1.2195 → 1.2185). Saving...
   Epoch 042/100 | Train: 1.2383 | Val: 1.2177 | LR: 7.35e-04 | Time: 1319.0s
      ✓ Val loss improved (1.2185 → 1.2177). Saving...
   Epoch 043/100 | Train: 1.2363 | Val: 1.2146 | LR: 7.19e-04 | Time: 1319.7s
      ✓ Val loss improved (1.2177 → 1.2146). Saving...
   Epoch 044/100 | Train: 1.2341 | Val: 1.2128 | LR: 7.04e-04 | Time: 1322.1s
      ✓ Val loss improved (1.2146 → 1.2128). Saving...
   Epoch 045/100 | Train: 1.2329 | Val: 1.2117 | LR: 6.88e-04 | Time: 1321.6s
      ✓ Val loss improved (1.2128 → 1.2117). Saving...
   Epoch 046/100 | Train: 1.2316 | Val: 1.2118 | LR: 6.71e-04 | Time: 1323.3s
      EarlyStopping counter: 1/10
   Epoch 047/100 | Train: 1.2306 | Val: 1.2095 | LR: 6.55e-04 | Time: 1323.9s
      ✓ Val 

In [16]:
# In[17]:

# =============================================================================
# CELL 17: SAVE FINAL RESULTS & PLOT
# =============================================================================

print("\n" + "="*70)
print("📊 FINAL RESULTS SUMMARY")
print("="*70)

results_json_path = os.path.join(CONFIG['results_dir'], "experiment_results.json")
with open(results_json_path, 'w') as f:
    json.dump(results, f, indent=4)
print(f"💾 Results saved: {results_json_path}")

print(f"\n   Error Model: {CONFIG['error_name']} ({CONFIG['platform']})")
print(f"   Eta: {CONFIG['eta']}, Vocab Size: {CONFIG['vocab_size']}")
print(f"   Seq Length: {CONFIG['seq_length']}")
print(f"   {'M':<8} {'Bi-LSTM':<12} {'Min.Dist':<12} {'KL Div':<12} {'Max.Like':<12}")
print(f"   {'-'*56}")
for i, M in enumerate(results['coverage']):
    print(f"   {M:<8} {results['lstm'][i]:<12.2f} {results['mindist'][i]:<12.2f} "
          f"{results['kl'][i]:<12.2f} {results['ml'][i]:<12.2f}")
print(f"   {'='*56}")

plot_path = os.path.join(CONFIG['results_dir'], "final_comparison_plot.png")
plot_comparison_results(results, plot_path, CONFIG)

print(f"\n✅ All experiments completed!")
print(f"   Error Model: {CONFIG['error_name']} ({CONFIG['platform']})")
print(f"   Eta: {CONFIG['eta']}, Classes: {CONFIG['vocab_size']}")
print(f"   Results directory: {CONFIG['results_dir']}")




📊 FINAL RESULTS SUMMARY
💾 Results saved: ./results_crossplatform_R21_eta0.2/experiment_results.json

   Error Model: R21 (Oxford Nanopore MinION)
   Eta: 0.2, Vocab Size: 34
   Seq Length: 136
   M        Bi-LSTM      Min.Dist     KL Div       Max.Like    
   --------------------------------------------------------
   1        7.55         7.64         7.64         7.64        
   2        11.46        10.37        10.37        10.37       
   3        14.15        11.95        11.95        11.95       
   5        17.69        14.29        13.59        13.59       
   8        22.90        17.15        17.21        17.21       
   10       25.65        18.17        17.95        17.95       
   15       31.48        20.45        20.41        20.41       
   20       36.25        22.00        22.03        22.03       
   25       40.34        23.06        23.19        23.19       
   30       43.90        24.06        24.44        24.44       
   40       49.73        25.26        25.6